# 08 - Robustness Sweep

Notebook 06's robustness check used one perturbation on one chip - a real number, but not a systematic answer to "evaluation of robustness under varying data conditions." This notebook sweeps the classical model across **multiple perturbation types x multiple flood events** (different geographies, different water/land ratios, different acquisition conditions) using `utils/ai/robustness/evaluate.py`'s `run_robustness_suite` - written earlier but never actually exercised until now.

Perturbations (`utils/ai/robustness/perturbations.py`): heavy speckle, a 10-degree incidence-angle shift, a texture-channel dropout (simulating a failed feature-computation stage - repurposed from the module's generic "secondary modality dropout" slot, since this pipeline's feature cube has no true optical channel), and two Gaussian noise levels.

In [ ]:
import os
import sys

sys.path.insert(0, next(
    d for d in (
        os.path.abspath(os.path.join(os.getcwd(), *([os.pardir] * i)))
        for i in range(8)
    )
    if os.path.exists(os.path.join(d, "pyproject.toml"))
))

import pickle

import numpy as np
import pandas as pd

from utils.ai.classic.sen1floods11_dataset import chip_id_from_s1_filename, load_split, read_label, read_s1
from utils.ai.robustness.evaluate import degradation_summary, run_robustness_suite, save_report
from utils.fusion.pixel_features import build_feature_cube
from utils.observability.run_logger import RunLogger

_root = sys.path[0]
logger = RunLogger("08_robustness_sweep")

SAR_CHANNELS = [0, 1]       # vv, vh
TEXTURE_CHANNELS = [4, 5]   # vv_local_mean, vv_local_std - stands in for the "secondary modality" slot

with open(os.path.join(_root, "datasets", "processed", "models", "classical_rf_v1.pkl"), "rb") as f:
    rf = pickle.load(f)

## Pick one chip from each of several distinct flood events

`sen1floods11` covers 11 real flood events across different geographies. Sampling across events (not just chips from one event) is what makes this "varying data conditions" rather than "varying noise on one scene."

In [2]:
N_EVENTS = 5

with logger.stage("select_chips_across_events") as stage:
    valid_pairs = load_split("valid")
    seen_events = {}
    for s1_filename, _ in valid_pairs:
        chip_id = chip_id_from_s1_filename(s1_filename)
        event = chip_id.split("_")[0]
        if event not in seen_events:
            seen_events[event] = chip_id
        if len(seen_events) >= N_EVENTS:
            break
    stage.metrics = {"events": list(seen_events.keys())}

print(f"sweeping {len(seen_events)} events: {list(seen_events.keys())}")

[08_robustness_sweep] -> select_chips_across_events ...
[08_robustness_sweep] <- select_chips_across_events [OK] 0.0s {'events': ['Ghana', 'India', 'Mekong', 'Nigeria', 'Pakistan']}
sweeping 5 events: ['Ghana', 'India', 'Mekong', 'Nigeria', 'Pakistan']


In [3]:
from utils.ai.classic.sen1floods11_dataset import s1_nodata_mask


def predict_chip_with_rf(cube: np.ndarray) -> np.ndarray:
    channels, h, w = cube.shape
    flat = cube.reshape(channels, h * w).T
    return rf.predict(flat).reshape(h, w)

all_results = {}
all_degradation = {}

for event, chip_id in seen_events.items():
    with logger.stage(f"robustness_sweep_{event}") as stage:
        s1 = read_s1(chip_id)
        label = read_label(chip_id)
        cube = build_feature_cube(s1)
        # combine the label's own -1 no-data convention with S1's independent
        # NaN nodata sentinel (confirmed NOT always coincident - see
        # utils/ai/classic/sen1floods11_dataset.py docstring)
        valid_mask = (label != -1) & ~s1_nodata_mask(s1)

        results = run_robustness_suite(
            predict_fn=predict_chip_with_rf,
            cube=cube,
            label=label,
            objective_name="flood-segmentation",
            sar_channel_indices=SAR_CHANNELS,
            optical_channel_indices=TEXTURE_CHANNELS,
            model_id=f"classical_rf_v1_{event}",
            valid_mask=valid_mask,
        )
        degradation = degradation_summary(results, primary_metric="iou")
        report_path = save_report(results)

        all_results[event] = results
        all_degradation[event] = degradation
        stage.metrics = {
            "chip_id": chip_id,
            "clean_iou": round(results["conditions"]["clean"]["iou"], 4),
            "n_nodata_s1_pixels": int(s1_nodata_mask(s1).sum()),
        }

print(f"{len(all_results)} events swept, reports saved to datasets/reports/")

[08_robustness_sweep] -> robustness_sweep_Ghana ...


[08_robustness_sweep] <- robustness_sweep_Ghana [OK] 4.935s {'chip_id': 'Ghana_5079', 'clean_iou': 1.0, 'n_nodata_s1_pixels': 0}
[08_robustness_sweep] -> robustness_sweep_India ...


[08_robustness_sweep] <- robustness_sweep_India [OK] 6.08s {'chip_id': 'India_1050276', 'clean_iou': 0.1847, 'n_nodata_s1_pixels': 0}
[08_robustness_sweep] -> robustness_sweep_Mekong ...


[08_robustness_sweep] <- robustness_sweep_Mekong [OK] 5.476s {'chip_id': 'Mekong_1149855', 'clean_iou': 0.4242, 'n_nodata_s1_pixels': 0}
[08_robustness_sweep] -> robustness_sweep_Nigeria ...


[08_robustness_sweep] <- robustness_sweep_Nigeria [OK] 7.24s {'chip_id': 'Nigeria_31096', 'clean_iou': 0.0384, 'n_nodata_s1_pixels': 0}
[08_robustness_sweep] -> robustness_sweep_Pakistan ...


[08_robustness_sweep] <- robustness_sweep_Pakistan [OK] 7.596s {'chip_id': 'Pakistan_43105', 'clean_iou': 0.0, 'n_nodata_s1_pixels': 733}
5 events swept, reports saved to datasets/reports/


## Degradation matrix: events x perturbations

Each cell is the fractional IoU drop relative to that event's own clean-condition IoU - real numbers, not averaged away, so a perturbation that only hurts certain geographies is visible rather than hidden.

In [4]:
degradation_df = pd.DataFrame(all_degradation).T
print(degradation_df.round(4))
print()
print("Mean degradation per perturbation across all events:")
print(degradation_df.mean(axis=0).round(4))

          speckle_heavy  incidence_shift_10deg  optical_dropout  \
Ghana            0.0000                 0.0000           0.0000   
India            0.5207                -0.1379          -1.1252   
Mekong           0.5540                -0.1812          -0.1454   
Nigeria          0.6894                -0.5184          -1.0633   
Pakistan            NaN                    NaN              NaN   

          gaussian_noise_low  gaussian_noise_high  
Ghana                 0.0000               0.0000  
India                 0.0987               0.3100  
Mekong                0.1657               0.7349  
Nigeria              -0.0226               0.8303  
Pakistan                 NaN                  NaN  

Mean degradation per perturbation across all events:
speckle_heavy            0.4410
incidence_shift_10deg   -0.2094
optical_dropout         -0.5835
gaussian_noise_low       0.0605
gaussian_noise_high      0.4688
dtype: float64


In [5]:
clean_ious = {event: r["conditions"]["clean"]["iou"] for event, r in all_results.items()}
logger.log_metrics({
    "events_swept": list(all_results.keys()),
    "clean_iou_by_event": clean_ious,
    "mean_degradation_by_perturbation": degradation_df.mean(axis=0).round(4).to_dict(),
})
logger.finalize()

[08_robustness_sweep] run complete in 32.726s -> D:\project-raw-data\sphoorthq-geoverse\datasets\reports\runs\cda821b2-6a71-4f45-9260-2838c21a6f97.json


'D:\\project-raw-data\\sphoorthq-geoverse\\datasets\\reports\\runs\\cda821b2-6a71-4f45-9260-2838c21a6f97.json'